# Sen4AgriNet: external crop data for Benguet

Prepare crop-reference labels and Sentinel-2 imagery for a balanced carrot/potato pilot. The workflow checks data, selects pixels, downloads cloud-screened composites and identifies **candidate green-up intervals**. Cabbage remains a target crop, but is absent from the inspected S4A subset.

**Audience:** students and researchers using Python 3.13. Main libraries are NumPy, netCDF4, Rasterio, Shapely, PyProj, Earth Engine and Matplotlib. Steps 6–8 use the accompanying `crop_workflow.py` and `observed_timelines.py` files.

**Inputs:** S4A `.nc` patches, generated `.geojson` footprints, sampling-mask and imagery `.tif` files, and saved JSON/CSV progress records. **Outputs:** spatial inventories, balanced pixel selections, 24 composites per region, quality reports, vegetation curves and unvalidated onset intervals.

## Before running

1. Open this notebook from the repository folder and select `.venv` as its kernel.
2. Copy `settings.example.json` to `settings.local.json` and set data paths and your Earth Engine project.
3. Run numbered sections in order. Package-installation cells are needed only once. Download and optional cells are labelled explicitly.
4. If raw S4A files already exist, skip Step 2’s preview/download cells. Step 3 still inspects the entire local 2020 dataset. Leave 2019 for future work.
5. Run all Step 5 settings and helper cells before inventory or downloads. On a new PC, follow [moving-PC guide](docs/moving_pc.md).

Steps 1–8 are implemented. Steps 9–15 describe the remaining work. Shared variables come from the preceding settings/results cells; do not run later sections in a fresh kernel without their prerequisites.

## Terms used here

- **Patch:** one S4A image file covering a small area.
- **Parcel/fragment:** a labeled field ID and the portion visible inside a patch. Fragments are not necessarily independent fields.
- **Granule:** a satellite image for one tile. Several granules can cover the same location and date.
- **GeoTIFF:** an image file that includes geographic coordinates.
- **SCL:** Sentinel-2’s scene classification layer, used to screen clouds, shadows and other invalid pixels.
- **Composite:** one summary image made from observations in a time interval. Here each pixel uses the median of valid daily values.
- **NDVI/NDRE/NDMI:** ratios of spectral bands that summarize vegetation or moisture-related signals. Step 7 calculates them explicitly.
- **Green-up:** a sustained vegetation increase, not a verified planting date.


## 1. Define the study

The study asks whether external crop-reference data can improve Random Forest classification in Benguet when verified local labels are limited.

| Setting | Plan |
|---|---|
| Crops | Cabbage, carrot and potato |
| Target area | Benguet; exact boundary and map reference period still need agreement |
| Imagery | Sentinel-2 L2A surface reflectance |
| Classifier window | Four consecutive months, with eight half-month intervals |
| Study bands | B3, B4, B5, B8 and B11 |
| Indices | NDVI, NDRE and NDMI |
| Feature comparison | 40 band features versus 64 band-plus-index features |
| Training comparison | External-only, local-only, and external plus the same local fields |
| Evaluation | The same independent Benguet test fields for all models |

The current development pilot uses 1,000 carrot pixels and 1,000 potato pixels. It is not a complete crop inventory. Pixels from the same parcel group must stay together in later splits.

The proposed window starts in the month containing accepted green-up. This rule still needs validation. Annual crop declarations do not prove which crop grew during a particular cycle. Confirm crop-cycle relevance before accepting training seasons.

Raw data and imagery stay on the configured data drive. Small reports live under `sen4agrinet_work`. Keep local paths in `settings.local.json`; that file is excluded from Git.


### Outputs and interpretation

A shared study scope and folder plan. Exact study boundaries, local label budgets, map dates and validation decisions remain open.


## 2. Download S4A files in batches

This step handles raw NetCDF downloads for **2020 only**. A saved file list and dataset revision keep batch membership reproducible.

If the files are already downloaded, run only the settings cell. Skip the preview and download cells. Step 3 discovers all local 2020 files independently of the selected download batch.

Set `dataset_root` to the folder containing `data`. For example, a root of `E:/ground_truth` resolves to `E:/ground_truth/data/2020`. Each collaborator supplies their own path.


### 2.1 Install packages — once

Skip this cell when the project environment already contains the required packages.


In [ ]:
# Install the two packages used in Step 2.
# Run this cell once in your notebook environment.
%pip install huggingface_hub netCDF4


### 2.2 Set paths and batch settings

Load local settings and select one download tile/batch. These settings do not restrict Step 3’s dataset-wide scan.


In [ ]:
from pathlib import Path
import json
import os

settings_path = Path("settings.local.json")
local_settings = (
    json.loads(settings_path.read_text(encoding="utf-8"))
    if settings_path.exists()
    else {}
)

# 1. Choose where the dataset is stored.
# This folder must contain the "data" folder.
# Replace this example with your own storage location.
dataset_folder = (
    Path(os.environ.get("S4A_ROOT", local_settings.get("dataset_root", "./S4A")))
    .expanduser()
    .resolve()
)

# 2. Choose where small inventory reports will be saved.
report_folder = Path(
    local_settings.get("report_root", "./sen4agrinet_work/inventory")
).resolve()

# 3. Select ONE tile and ONE batch.
dataset_name = "orion-ai-lab/S4A"
year = 2020  # Keep 2019 for future work.
tile = "31TBF"
files_per_batch = 50
batch_number = 1  # Batch numbers start at 1.

if year != 2020:
    raise ValueError("Step 2 currently processes 2020 only.")
if files_per_batch < 1 or batch_number < 1:
    raise ValueError("Batch size and batch number must be at least 1.")

print("Dataset folder:", dataset_folder)
print("Selected tile:", tile)
print("Selected batch:", batch_number)


### 2.3 Preview one batch — optional

Save/reuse the remote file list and display the selected batch. This requests metadata, not imagery.


In [ ]:
import json
from huggingface_hub import HfApi

# Save one file list for each tile so batch membership stays consistent.
tile_report_folder = report_folder / str(year) / tile
tile_report_folder.mkdir(parents=True, exist_ok=True)
file_list_path = tile_report_folder / "download_plan.json"

if file_list_path.exists():
    # Reuse the saved file list and dataset version.
    download_plan = json.loads(file_list_path.read_text(encoding="utf-8"))
    if (
        download_plan["repo_id"] != dataset_name
        or download_plan["year"] != year
        or download_plan["tile"] != tile
    ):
        raise ValueError("The saved file list belongs to a different dataset or tile.")
    if download_plan["batch_size"] != files_per_batch:
        raise ValueError(
            "Use the saved batch size or choose a different report folder."
        )
else:
    # Request filenames and sizes only. This does not download the imagery.
    hub = HfApi()
    dataset_version = hub.repo_info(dataset_name, repo_type="dataset").sha
    remote_tile_folder = f"data/{year}/{tile}"
    remote_items = hub.list_repo_tree(
        repo_id=dataset_name,
        repo_type="dataset",
        revision=dataset_version,
        path_in_repo=remote_tile_folder,
        recursive=True,
    )

    available_files = []
    for item in remote_items:
        is_netcdf_file = item.path.endswith(".nc") and hasattr(item, "size")
        if is_netcdf_file:
            available_files.append({"filename": item.path, "expected_bytes": item.size})

    available_files.sort(key=lambda file: file["filename"])
    if not available_files:
        raise ValueError("No NetCDF files were found in the selected tile.")

    download_plan = {
        "repo_id": dataset_name,
        "revision": dataset_version,
        "year": year,
        "tile": tile,
        "batch_size": files_per_batch,
        "files": available_files,
    }
    file_list_path.write_text(json.dumps(download_plan, indent=2), encoding="utf-8")

# Pick only the filenames belonging to the selected batch.
first_file = (batch_number - 1) * files_per_batch
last_file = first_file + files_per_batch
batch_files = download_plan["files"][first_file:last_file]

if not batch_files:
    raise ValueError("This batch number is beyond the end of the file list.")

batch_name = f"s4a_{year}_{tile}_batch_{batch_number:03d}"
number_of_batches = (
    len(download_plan["files"]) + files_per_batch - 1
) // files_per_batch
batch_bytes = sum(file["expected_bytes"] for file in batch_files)

print("Batch:", batch_number, "of", number_of_batches)
print("Files in this batch:", len(batch_files))
print("Full batch size (GiB):", round(batch_bytes / 1024**3, 2))
print("Dataset version:", download_plan["revision"])

for file in batch_files:
    local_file = dataset_folder / file["filename"]
    if local_file.is_file() and local_file.stat().st_size == file["expected_bytes"]:
        print("Already present:", file["filename"])
    else:
        print("Needs download:", file["filename"])


### 2.4 Download one batch — optional

Retrieve missing or size-mismatched files, check headers and save progress after each file.


In [ ]:
import csv
from datetime import datetime, timezone
from huggingface_hub import hf_hub_download
from netCDF4 import Dataset

# This cell processes ONLY the batch shown in the previous cell.
manifest_path = tile_report_folder / f"{batch_name}.csv"
report_columns = [
    "batch",
    "dataset",
    "version",
    "year",
    "tile",
    "filename",
    "local_path",
    "expected_bytes",
    "actual_bytes",
    "checked_at_utc",
    "status",
    "error",
]

# Preserve previous results if this batch was interrupted.
results_by_filename = {}
if manifest_path.exists():
    with manifest_path.open(newline="", encoding="utf-8") as report:
        for row in csv.DictReader(report):
            results_by_filename[row["filename"]] = row


def save_progress():
    """Save results after every file without leaving a half-written report."""
    temporary_path = manifest_path.with_suffix(".csv.tmp")
    with temporary_path.open("w", newline="", encoding="utf-8") as report:
        writer = csv.DictWriter(report, fieldnames=report_columns)
        writer.writeheader()
        for file in batch_files:
            if file["filename"] in results_by_filename:
                writer.writerow(results_by_filename[file["filename"]])
    temporary_path.replace(manifest_path)


for file in batch_files:
    filename = file["filename"]
    local_file = dataset_folder / filename
    expected_size = file["expected_bytes"]

    result = {
        "batch": batch_name,
        "dataset": dataset_name,
        "version": download_plan["revision"],
        "year": year,
        "tile": tile,
        "filename": filename,
        "local_path": str(local_file),
        "expected_bytes": expected_size,
        "actual_bytes": "",
        "checked_at_utc": datetime.now(timezone.utc).isoformat(),
        "status": "pending",
        "error": "",
    }

    try:
        # Download missing files or replace files with the wrong size.
        needs_download = True
        if local_file.is_file():
            needs_download = local_file.stat().st_size != expected_size

        if needs_download:
            print("Downloading:", filename)
            hf_hub_download(
                repo_id=dataset_name,
                repo_type="dataset",
                revision=download_plan["revision"],
                filename=filename,
                local_dir=dataset_folder,
                force_download=local_file.exists(),
            )

        # Check size, then open the header without loading imagery arrays.
        result["actual_bytes"] = local_file.stat().st_size
        if result["actual_bytes"] != expected_size:
            raise ValueError("The local file size does not match the remote file size.")

        with Dataset(local_file, "r") as netcdf_file:
            # Access metadata only. The context manager closes the file.
            group_names = list(netcdf_file.groups)

        result["status"] = (
            "downloaded_header_readable"
            if needs_download
            else "existing_header_readable"
        )

    except Exception as error:
        # Record the problem and continue with the other files in this batch.
        result["status"] = "failed"
        result["error"] = f"{type(error).__name__}: {error}"

    results_by_filename[filename] = result
    save_progress()
    print(filename, "->", result["status"])
    if result["error"]:
        print("  Reason:", result["error"])

failed_files = 0
for result in results_by_filename.values():
    if result["status"] == "failed":
        failed_files += 1

print("Report saved to:", manifest_path)
print("Failed files:", failed_files)


### Outputs and interpretation

`download_plan.json` stores a tile’s file list and revision. A per-batch CSV records sizes, header checks and errors. Raw files stay under the configured `data/2020/<tile>/` folder. A readable header is not a complete integrity check.


## 3. Inspect the downloaded 2020 dataset

Find crop labels, parcel IDs, required bands and observation dates in every downloaded patch. Read masks and small date arrays; do not load the spectral time series.

Original S4A codes are cabbage 213, carrot 231 and potato 510. The report lists all observed codes, but cannot automatically prove that the chosen encoding is correct.

Files are read one at a time. Checkpoints reuse successful results only when the file and inspection settings match. Failed files and warning-bearing files are checked again. Set `repeat_checks = True` to repeat every check.

The scan covers local files, not guaranteed completeness against the remote release. Parcel counts are occurrences across files, not deduplicated independent fields.


### 3.1 Choose inspection scope

Find every local 2020 NetCDF file and set the label codes and report folder.


In [ ]:
import numpy as np
from netCDF4 import Dataset, num2date

# Inspect ALL downloaded 2020 tiles, regardless of the Step 2 batch selection.
inspection_year = 2020
raw_year_folder = dataset_folder / "data" / str(inspection_year)
if not raw_year_folder.is_dir():
    raise FileNotFoundError(f"Dataset folder does not exist: {raw_year_folder}")

# Read files sequentially; the file list itself is small metadata.
dataset_files = sorted(raw_year_folder.rglob("*.nc"))
if not dataset_files:
    raise ValueError("No downloaded NetCDF files were found for 2020.")

crop_codes = {"cabbage": 213, "carrot": 231, "potato": 510}
required_bands = ["B03", "B04", "B05", "B08", "B11"]
parcel_background_ids = [0, -1]
repeat_checks = False

inspection_folder = report_folder / str(inspection_year) / "dataset_inspection"
inspection_folder.mkdir(parents=True, exist_ok=True)
print("Dataset year:", inspection_year)
print("Local files found:", len(dataset_files))
print("Reports:", inspection_folder)


### 3.2 Define inspection helpers

These functions read masks, find nested variables and decode dates. Definitions do not run a dataset scan.


In [ ]:
def collect_variables(group):
    """Find variables inside the root and all NetCDF subgroups."""
    variables = []
    for variable in group.variables.values():
        variables.append(variable)
    for child_group in group.groups.values():
        variables.extend(collect_variables(child_group))
    return variables


def find_one_variable(variables, name):
    matches = [
        variable for variable in variables if variable.name.lower() == name.lower()
    ]
    if len(matches) != 1:
        raise ValueError(f"Expected one {name} variable; found {len(matches)}.")
    return matches[0]


def read_mask(variable):
    """Read a 2D integer mask while preserving invalid-pixel information."""
    masked_values = np.ma.asarray(variable[:])
    if masked_values.ndim != 2:
        raise ValueError(f"{variable.name} is not a 2D mask: {masked_values.shape}")
    values = np.asarray(masked_values.data)
    valid = ~np.ma.getmaskarray(masked_values)
    valid = valid & np.isfinite(values)
    if np.any(values[valid] != np.floor(values[valid])):
        raise ValueError(f"{variable.name} contains noninteger values.")
    return values, valid


def inspect_dates(variable):
    """Decode dates without guessing units or counting duplicate dates twice."""
    units = getattr(variable, "units", "")
    if "since" not in units:
        raise ValueError("Time coordinate lacks CF 'since' units.")
    time_values = np.ma.asarray(variable[:]).compressed()
    decoded_dates = num2date(
        time_values, units, calendar=getattr(variable, "calendar", "standard")
    )
    unique_dates = sorted(set(decoded_dates))
    gaps = []
    for earlier, later in zip(unique_dates, unique_dates[1:]):
        gaps.append((later - earlier).total_seconds() / 86400)
    return {
        "first_date": str(unique_dates[0]) if unique_dates else None,
        "last_date": str(unique_dates[-1]) if unique_dates else None,
        "unique_observations": len(unique_dates),
        "largest_gap_days": max(gaps) if gaps else None,
        "dates": [str(date) for date in unique_dates],
    }


def inspect_file(local_file):
    """Inspect one file, then close it before processing another."""
    details = {"status": "ok", "warnings": [], "crops": {}, "bands": {}, "times": []}
    with Dataset(local_file, "r") as dataset:
        variables = collect_variables(dataset)
        details["metadata"] = {
            name: str(dataset.getncattr(name)) for name in dataset.ncattrs()
        }
        label_variable = find_one_variable(variables, "labels")
        labels, valid_labels = read_mask(label_variable)
        details["label_shape"] = list(labels.shape)
        codes, counts = np.unique(labels[valid_labels], return_counts=True)
        details["observed_codes"] = {
            str(int(code)): int(count) for code, count in zip(codes, counts)
        }

        parcels = None
        valid_parcels = None
        try:
            parcels, valid_parcels = read_mask(find_one_variable(variables, "parcels"))
            if parcels.shape != labels.shape:
                raise ValueError("Label and parcel masks have different shapes.")
            valid_parcels = valid_parcels & ~np.isin(parcels, parcel_background_ids)
        except Exception as error:
            parcels = None
            details["warnings"].append(f"Parcel mask: {error}")

        for crop, code in crop_codes.items():
            crop_pixels = valid_labels & (labels == code)
            crop_result = {
                "code": code,
                "pixels": int(crop_pixels.sum()),
                "parcel_occurrences": None,
                "parcel_ids": None,
                "pixels_without_valid_parcel": None,
            }
            if parcels is not None:
                parcel_ids = np.unique(parcels[crop_pixels & valid_parcels])
                crop_result["parcel_ids"] = [int(parcel_id) for parcel_id in parcel_ids]
                crop_result["parcel_occurrences"] = len(parcel_ids)
                crop_result["pixels_without_valid_parcel"] = int(
                    (crop_pixels & ~valid_parcels).sum()
                )
            details["crops"][crop] = crop_result

        # Check band shapes, never read the spectral arrays here.
        for band in required_bands:
            matches = [
                variable for variable in variables if variable.name.upper() == band
            ]
            details["bands"][band] = [
                {"shape": list(variable.shape), "dimensions": list(variable.dimensions)}
                for variable in matches
            ]
            if len(matches) != 1:
                details["warnings"].append(
                    f"Band {band}: expected one array; found {len(matches)}."
                )

        # Keep each band's time coordinate separate: dates can differ by band.
        for variable in variables:
            if variable.name.lower() not in ("time", "timestamp"):
                continue
            time_result = {
                "variable": variable.group().path.rstrip("/") + "/" + variable.name
            }
            try:
                time_result.update(inspect_dates(variable))
            except Exception as error:
                time_result["error"] = str(error)
                details["warnings"].append(f"{time_result['variable']}: {error}")
            details["times"].append(time_result)
        if not details["times"]:
            details["warnings"].append("No time or timestamp coordinate found.")
    return details


### 3.3 Inspect files sequentially

Process one file at a time and save/reuse per-file checkpoints. Errors are recorded without treating failed files as crop absence.


In [ ]:
import csv
import json
from datetime import datetime, timezone

inspection_settings = {
    "crop_codes": crop_codes,
    "bands": required_bands,
    "parcel_background_ids": parcel_background_ids,
    "inspection_version": 1,
}
inspection_records = []

for file_number, local_file in enumerate(dataset_files, start=1):
    filename = local_file.relative_to(dataset_folder).as_posix()
    # Each file gets its own checkpoint, so interruption does not lose earlier work.
    relative_path = local_file.relative_to(raw_year_folder)
    checkpoint_path = (
        inspection_folder / "checkpoints" / relative_path.with_suffix(".nc.json")
    )
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    signature = None
    if local_file.is_file():
        file_info = local_file.stat()
        signature = {"bytes": file_info.st_size, "modified_ns": file_info.st_mtime_ns}

    previous_result = None
    if checkpoint_path.exists() and not repeat_checks:
        try:
            previous_result = json.loads(checkpoint_path.read_text(encoding="utf-8"))
        except (ValueError, OSError):
            previous_result = None

    can_reuse = (
        previous_result is not None
        and signature is not None
        and previous_result.get("signature") == signature
        and previous_result.get("settings") == inspection_settings
        and previous_result.get("status") == "ok"
        and not previous_result.get("warnings")
    )
    if can_reuse:
        result = previous_result
    else:
        result = {
            "filename": filename,
            "signature": signature,
            "settings": inspection_settings,
            "checked_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        try:
            if signature is None:
                raise FileNotFoundError(
                    "File is missing. Refresh the file list and check Step 2 downloads."
                )
            result.update(inspect_file(local_file))
        except Exception as error:
            result.update(status="failed", error=f"{type(error).__name__}: {error}")
        temporary_path = checkpoint_path.with_suffix(".json.tmp")
        temporary_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
        temporary_path.replace(checkpoint_path)

    inspection_records.append(result)
    print(
        f"[{file_number}/{len(dataset_files)}] {Path(filename).name}: {result['status']}"
    )
    if result["status"] == "ok":
        for crop, crop_result in result["crops"].items():
            print(
                f"  {crop}: {crop_result['pixels']} pixels; {crop_result['parcel_occurrences']} parcel occurrences"
            )
        for warning in result["warnings"]:
            print("  Check:", warning)
    else:
        print("  Reason:", result["error"])



### 3.4 Write dataset reports

Use the inspection records from the previous cell to build counts and the Step 4 candidate list.


In [ ]:
# Build a dataset-wide table from all files selected for this run.
columns = [
    "filename",
    "crop",
    "code",
    "pixels",
    "parcel_occurrences",
    "pixels_without_valid_parcel",
    "missing_bands",
    "warnings",
]
with (inspection_folder / "crop_counts.csv").open(
    "w", newline="", encoding="utf-8"
) as report:
    writer = csv.DictWriter(report, fieldnames=columns)
    writer.writeheader()
    for result in inspection_records:
        if result["status"] != "ok":
            continue
        missing_bands = [band for band, arrays in result["bands"].items() if not arrays]
        for crop, counts in result["crops"].items():
            writer.writerow(
                {
                    "filename": result["filename"],
                    "crop": crop,
                    "code": counts["code"],
                    "pixels": counts["pixels"],
                    "parcel_occurrences": counts["parcel_occurrences"],
                    "pixels_without_valid_parcel": counts[
                        "pixels_without_valid_parcel"
                    ],
                    "missing_bands": "; ".join(missing_bands),
                    "warnings": "; ".join(result["warnings"]),
                }
            )

successful_files = [result for result in inspection_records if result["status"] == "ok"]
candidate_patch_paths = [
    result["filename"]
    for result in successful_files
    if any(counts["pixels"] > 0 for counts in result["crops"].values())
]
crop_inventory_summary = {
    "year": inspection_year,
    "input_folder": str(raw_year_folder),
    "coverage": "all locally downloaded 2020 NetCDF files; remote completeness unverified",
    "settings": inspection_settings,
    "files_selected": len(inspection_records),
    "labels_read_successfully": len(successful_files),
    "failed_files": [
        result["filename"]
        for result in inspection_records
        if result["status"] == "failed"
    ],
    "candidate_files": candidate_patch_paths,
    "crops": {},
}
for crop in crop_codes:
    counts = [result["crops"][crop] for result in successful_files]
    crop_inventory_summary["crops"][crop] = {
        "pixels": sum(item["pixels"] for item in counts),
        "files_containing_crop": sum(item["pixels"] > 0 for item in counts),
        "known_parcel_occurrences": sum(
            item["parcel_occurrences"]
            for item in counts
            if item["parcel_occurrences"] is not None
        ),
        "files_missing_parcel_counts": sum(
            item["parcel_occurrences"] is None for item in counts
        ),
    }
(inspection_folder / "summary.json").write_text(
    json.dumps(crop_inventory_summary, indent=2), encoding="utf-8"
)
print("Reports saved to:", inspection_folder)
print("Candidate files for Step 4:", len(candidate_patch_paths))
print("Parcel occurrences are not unique independent fields.")


### Outputs and interpretation

`crop_counts.csv` gives pixel and parcel-occurrence counts. `summary.json` lists coverage, failures and candidates. Per-file JSON checkpoints preserve parcel IDs, band metadata and dates. Zero counts apply only to successfully inspected labels under the chosen encoding.


## 4. Review fields and prepare the balanced pilot

Use Step 3's candidate patches to check image alignment and recover geographic crop fragments. Verify each source coordinate system and pixel grid before extracting polygons. Do not guess a coordinate system when metadata is missing.

Review one patch and date at a time. RGB previews use percentile stretching for display and remain unmasked S4A L1C imagery. They do not independently verify the declared crop.

Keep uncertain field fragments separate. Remove only redundant footprints with matching country, year, crop, parcel ID and geometry. Preserve their source records and report remaining repeated IDs and overlaps.


### 4.1 Install geometry packages — once

Skip when these packages are already installed in the project environment.


In [ ]:
%pip install matplotlib rasterio shapely pyproj


### 4.2 Load candidates and review settings

Choose one patch/date for display. Boundary extraction later covers all candidate patches.


In [ ]:
import json
import csv
import hashlib
import numpy as np
import matplotlib.pyplot as plt
from netCDF4 import Dataset, num2date
from affine import Affine
from rasterio.features import shapes
from pyproj import CRS, Transformer
from shapely.geometry import shape, mapping
from shapely.ops import unary_union, transform as transform_geometry
from shapely.strtree import STRtree

summary_path = inspection_folder / "summary.json"
inspection_summary = json.loads(summary_path.read_text(encoding="utf-8"))
candidate_files = inspection_summary["candidate_files"]
if not candidate_files:
    raise ValueError("Step 3 found no target-crop candidates.")

field_folder = report_folder.parent / "fields" / "2020"
field_folder.mkdir(parents=True, exist_ok=True)
candidate_number = 1  # One-based patch selection for visual review.
acquisition_number = 1  # One-based date selection; inspect several dates.
review_status = "unreviewed"  # Use unreviewed, needs_followup or overlay_checked.
review_note = ""  # Describe alignment, clouds and suspicious labels.
print("Candidate patches:", len(candidate_files))


### 4.3 Define grid and boundary helpers

Validate coordinates and extract target-crop polygons while preserving provenance.


In [ ]:
def read_grid(variable):
    """Validate the CRS and pixel-center coordinates before polygonizing."""
    group = variable.group()
    if not hasattr(variable, "crs") or not hasattr(variable, "transform"):
        raise ValueError("Missing source CRS or affine transform.")
    crs_text = str(variable.crs).replace("+init=epsg:", "EPSG:")
    source_crs = CRS.from_user_input(crs_text)
    coefficients = np.asarray(variable.transform).ravel()
    if len(coefficients) != 6:
        raise ValueError("Expected six affine coefficients.")
    grid_transform = Affine(*coefficients)
    if grid_transform.b != 0 or grid_transform.d != 0:
        raise ValueError("Rotated grids need an explicit coordinate validation method.")
    x = np.asarray(group.variables["x"][:])
    y = np.asarray(group.variables["y"][:])
    if tuple(variable.shape[-2:]) != (len(y), len(x)):
        raise ValueError("Coordinate lengths differ from raster shape.")
    expected_x = grid_transform.c + (np.arange(len(x)) + 0.5) * grid_transform.a
    expected_y = grid_transform.f + (np.arange(len(y)) + 0.5) * grid_transform.e
    if not np.allclose(x, expected_x, rtol=0, atol=0.001) or not np.allclose(
        y, expected_y, rtol=0, atol=0.001
    ):
        raise ValueError("Pixel centers disagree with the stored affine transform.")
    return source_crs, grid_transform


def extract_fragments(local_file, source_filename):
    """Extract target-crop fragments while keeping original provenance."""
    features = []
    with Dataset(local_file, "r") as dataset:
        variables = collect_variables(dataset)
        label_variable = find_one_variable(variables, "labels")
        parcel_variable = find_one_variable(variables, "parcels")
        labels, valid_labels = read_mask(label_variable)
        parcels, valid_parcels = read_mask(parcel_variable)
        label_crs, label_transform = read_grid(label_variable)
        parcel_crs, parcel_transform = read_grid(parcel_variable)
        if (
            labels.shape != parcels.shape
            or label_crs != parcel_crs
            or label_transform != parcel_transform
        ):
            raise ValueError("Label and parcel grids do not match.")
        valid_parcels &= ~np.isin(parcels, parcel_background_ids)
        to_wgs84 = Transformer.from_crs(label_crs, "EPSG:4326", always_xy=True)

        for crop, code in crop_codes.items():
            selected_crop = valid_labels & (labels == code)
            parcel_ids = np.unique(parcels[selected_crop & valid_parcels])
            for parcel_id in parcel_ids:
                selected = selected_crop & valid_parcels & (parcels == parcel_id)
                pieces = []
                for geometry, value in shapes(
                    selected.astype("uint8"), mask=selected, transform=label_transform
                ):
                    if value == 1:
                        pieces.append(shape(geometry))
                geometry = unary_union(pieces)
                if geometry.is_empty or not geometry.is_valid:
                    raise ValueError(f"Invalid geometry for parcel {int(parcel_id)}.")
                geographic_geometry = transform_geometry(to_wgs84.transform, geometry)
                if not geographic_geometry.is_valid:
                    raise ValueError("Reprojected geometry is invalid.")
                west, south, east, north = geographic_geometry.bounds
                if not (-180 <= west <= east <= 180 and -90 <= south <= north <= 90):
                    raise ValueError(
                        "Reprojected coordinates are outside WGS84 bounds."
                    )
                source_key = f"{source_filename}|{crop}|{int(parcel_id)}"
                fragment_id = hashlib.sha256(source_key.encode()).hexdigest()[:20]
                other_labels = (
                    valid_labels
                    & valid_parcels
                    & (parcels == parcel_id)
                    & (labels != code)
                )
                touches_edge = bool(
                    selected[0].any()
                    or selected[-1].any()
                    or selected[:, 0].any()
                    or selected[:, -1].any()
                )
                properties = {
                    "fragment_id": fragment_id,
                    "field_id": None,
                    "source_file": source_filename,
                    "year": str(getattr(dataset, "patch_year", "unknown")),
                    "country": str(getattr(dataset, "patch_country_code", "unknown")),
                    "tile": str(getattr(dataset, "patch_tile", "unknown")),
                    "crop": crop,
                    "crop_code": code,
                    "parcel_id": int(parcel_id),
                    "crop_pixels": int(selected.sum()),
                    "mixed_label_parcel": bool(other_labels.any()),
                    "touches_patch_edge": touches_edge,
                    "source_crs": label_crs.to_string(),
                    "review_status": "unreviewed",
                    "geometry_status": "candidate_fragment",
                }
                features.append(
                    {
                        "type": "Feature",
                        "properties": properties,
                        "geometry": mapping(geographic_geometry),
                    }
                )
    return features


### 4.4 Inspect RGB and outlines

Display one dated image and save the chosen review status/note. Change the selection to inspect other patches or dates.


In [ ]:
# Review one patch/date without loading its full time series.
if not 1 <= candidate_number <= len(candidate_files):
    raise ValueError("Candidate number is outside the candidate list.")
selected_filename = candidate_files[candidate_number - 1]
selected_path = dataset_folder / selected_filename

with Dataset(selected_path, "r") as dataset:
    variables = collect_variables(dataset)
    label_variable = find_one_variable(variables, "labels")
    labels, valid_labels = read_mask(label_variable)
    parcels, valid_parcels = read_mask(find_one_variable(variables, "parcels"))
    label_crs, label_transform = read_grid(label_variable)
    channels = []
    reference_dates = None
    for band in ["B04", "B03", "B02"]:
        variable = find_one_variable(variables, band)
        band_crs, band_transform = read_grid(variable)
        if (
            variable.shape[-2:] != labels.shape
            or band_crs != label_crs
            or band_transform != label_transform
        ):
            raise ValueError(
                "RGB and label grids differ; explicit resampling is required."
            )
        time_variable = variable.group().variables["time"]
        dates = list(
            num2date(
                time_variable[:],
                time_variable.units,
                calendar=getattr(time_variable, "calendar", "standard"),
            )
        )
        if reference_dates is not None and dates != reference_dates:
            raise ValueError("RGB acquisition dates differ between bands.")
        reference_dates = dates
        if not 1 <= acquisition_number <= len(dates):
            raise ValueError(f"Acquisition must be between 1 and {len(dates)}.")
        values = np.ma.asarray(variable[acquisition_number - 1], dtype=float).filled(
            np.nan
        )
        finite_values = values[np.isfinite(values)]
        if not len(finite_values):
            raise ValueError(f"{band} has no valid pixels for this date.")
        low, high = np.percentile(finite_values, [2, 98])
        channels.append(
            np.nan_to_num(np.clip((values - low) / max(high - low, 1e-9), 0, 1))
        )
    rgb = np.stack(channels, axis=-1)
    date_text = str(reference_dates[acquisition_number - 1])

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for axis in axes:
    axis.imshow(rgb)
    axis.set_axis_off()
axes[0].set_title("Exploratory RGB (unmasked)")
colors = {"cabbage": "cyan", "carrot": "magenta", "potato": "orange"}
for crop, code in crop_codes.items():
    selected = valid_labels & (labels == code)
    if selected.any() and not selected.all():
        axes[1].contour(
            selected.astype(int), levels=[0.5], colors=colors[crop], linewidths=1
        )
        axes[1].plot([], [], color=colors[crop], label=crop)
# Show parcel transitions only where both neighbouring parcel pixels are valid.
edges = np.zeros(labels.shape, dtype=bool)
edges[:-1] |= (parcels[:-1] != parcels[1:]) & valid_parcels[:-1] & valid_parcels[1:]
edges[:, :-1] |= (
    (parcels[:, :-1] != parcels[:, 1:]) & valid_parcels[:, :-1] & valid_parcels[:, 1:]
)
edge_overlay = np.zeros((*labels.shape, 4))
edge_overlay[edges] = [1, 1, 1, 0.6]
axes[1].imshow(edge_overlay)
axes[1].legend()
axes[1].set_title("Target crop outlines and white parcel edges")
fig.suptitle(f"{Path(selected_filename).name} | {date_text}")
plt.show()

# Edit review_status/review_note in settings and rerun this cell to save a review.
if review_status not in ["unreviewed", "needs_followup", "overlay_checked"]:
    raise ValueError("Unknown review status.")
review_path = field_folder / "visual_reviews.json"
reviews = (
    json.loads(review_path.read_text(encoding="utf-8")) if review_path.exists() else {}
)
reviews[selected_filename + "|" + date_text] = {
    "source_file": selected_filename,
    "date": date_text,
    "status": review_status,
    "note": review_note,
}
review_path.write_text(json.dumps(reviews, indent=2), encoding="utf-8")
print("Review notes saved:", review_path)


### 4.5 Extract all fragments

Checkpoint geographic boundaries and report overlapping or repeated-ID relationships. Do not automatically merge them.


In [ ]:
# Extract all Step 3 candidates, one file at a time.
checkpoint_folder = field_folder / "patch_fragments"
checkpoint_folder.mkdir(exist_ok=True)
all_features = []
errors = []
for number, filename in enumerate(candidate_files, start=1):
    local_file = dataset_folder / filename
    checkpoint_name = hashlib.sha256(filename.encode()).hexdigest() + ".json"
    checkpoint_path = checkpoint_folder / checkpoint_name
    try:
        stat = local_file.stat()
        signature = {
            "file": filename,
            "bytes": stat.st_size,
            "modified_ns": stat.st_mtime_ns,
            "crop_codes": crop_codes,
            "background": parcel_background_ids,
            "version": 1,
        }
        saved = (
            json.loads(checkpoint_path.read_text(encoding="utf-8"))
            if checkpoint_path.exists()
            else None
        )
        if saved and saved.get("signature") == signature:
            features = saved["features"]
        else:
            features = extract_fragments(local_file, filename)
            temporary_path = checkpoint_path.with_suffix(".tmp")
            temporary_path.write_text(
                json.dumps({"signature": signature, "features": features}),
                encoding="utf-8",
            )
            temporary_path.replace(checkpoint_path)
        all_features.extend(features)
        print(
            f"[{number}/{len(candidate_files)}] {Path(filename).name}: {len(features)} fragments"
        )
    except Exception as error:
        errors.append(
            {"source_file": filename, "error": f"{type(error).__name__}: {error}"}
        )
        print("Needs investigation:", filename, str(error))

collection = {"type": "FeatureCollection", "features": all_features}
(field_folder / "candidate_fragments.geojson").write_text(
    json.dumps(collection), encoding="utf-8"
)
(field_folder / "extraction_errors.json").write_text(
    json.dumps(errors, indent=2), encoding="utf-8"
)

# Use a spatial index to find intersections without comparing every pair.
geometries = [shape(feature["geometry"]) for feature in all_features]
overlap_rows = []
if geometries:
    spatial_index = STRtree(geometries)
    for first_index, geometry in enumerate(geometries):
        for second_index in spatial_index.query(geometry, predicate="intersects"):
            second_index = int(second_index)
            if second_index <= first_index:
                continue
            first = all_features[first_index]["properties"]
            second = all_features[second_index]["properties"]
            relation = (
                "exact_duplicate"
                if geometry.equals(geometries[second_index])
                else "overlap_or_boundary_contact"
            )
            overlap_rows.append(
                {
                    "first_fragment": first["fragment_id"],
                    "second_fragment": second["fragment_id"],
                    "relation": relation,
                    "crop_conflict": first["crop"] != second["crop"],
                }
            )
with (field_folder / "spatial_review.csv").open(
    "w", newline="", encoding="utf-8"
) as report:
    writer = csv.DictWriter(
        report,
        fieldnames=["first_fragment", "second_fragment", "relation", "crop_conflict"],
    )
    writer.writeheader()
    writer.writerows(overlap_rows)

# Repeated IDs are review hints, not evidence of a shared independent field.
id_groups = {}
for feature in all_features:
    properties = feature["properties"]
    key = (properties["country"], properties["year"], properties["parcel_id"])
    id_groups.setdefault(key, []).append(properties["fragment_id"])
repeated_ids = [
    {"country": key[0], "year": key[1], "parcel_id": key[2], "fragments": ids}
    for key, ids in id_groups.items()
    if len(ids) > 1
]
(field_folder / "repeated_parcel_ids.json").write_text(
    json.dumps(repeated_ids, indent=2), encoding="utf-8"
)
print("Candidate fragments:", len(all_features))
print("Extraction failures:", len(errors))
print("Spatial relationships needing review:", len(overlap_rows))
print("Output folder:", field_folder)
print("Independent-field IDs remain unset until deduplication is verified.")


### 4.6 Remove verified redundant footprints

Keep the original file and save a separate retrieval list with duplicate-to-retained links.


In [ ]:
# Deduplicate before Step 5. Original candidate fragments are preserved.
# Matching IDs alone are insufficient: geometry and crop must also match.
from collections import defaultdict
from shapely.geometry import shape
from shapely.strtree import STRtree
import json
import csv

original_path = field_folder / "candidate_fragments.geojson"
original_features = json.loads(original_path.read_text(encoding="utf-8"))["features"]
original_features.sort(key=lambda feature: feature["properties"]["fragment_id"])
retained_features = []
groups = defaultdict(list)
duplicate_records = []

for feature in original_features:
    properties = feature["properties"]
    # IDs are scoped by country and year; geometry proves the redundant footprint.
    key = (
        properties["country"],
        properties["year"],
        properties["crop"],
        properties["parcel_id"],
    )
    geometry = shape(feature["geometry"])
    if geometry.is_empty or not geometry.is_valid:
        raise ValueError("Invalid geometry must be resolved before deduplication.")
    matching_feature = None
    for previous in groups[key]:
        if geometry.equals(shape(previous["geometry"])):
            matching_feature = previous
            break
    if matching_feature is None:
        feature["properties"]["duplicate_sources"] = []
        groups[key].append(feature)
        retained_features.append(feature)
    else:
        duplicate_record = {
            "removed_fragment": properties["fragment_id"],
            "retained_fragment": matching_feature["properties"]["fragment_id"],
            "source_file": properties["source_file"],
            "reason": "Same country, year, crop, parcel ID and equal WGS84 geometry",
        }
        duplicate_records.append(duplicate_record)
        matching_feature["properties"]["duplicate_sources"].append(dict(properties))

# Remaining repeated IDs may be partial fragments. Do not discard or merge them.
remaining_ids = defaultdict(list)
for feature in retained_features:
    properties = feature["properties"]
    key = (properties["country"], properties["year"], properties["parcel_id"])
    remaining_ids[key].append(properties)
unresolved_ids = []
for key, properties in remaining_ids.items():
    if len(properties) > 1:
        unresolved_ids.append(
            {
                "country": key[0],
                "year": key[1],
                "parcel_id": key[2],
                "fragment_ids": [item["fragment_id"] for item in properties],
                "crops": sorted(set(item["crop"] for item in properties)),
                "status": "Different footprints; verify ID scope and field fragments before merging",
            }
        )

# Record remaining spatial relationships, including different-ID overlaps.
geometries = [shape(feature["geometry"]) for feature in retained_features]
spatial_rows = []
if geometries:
    tree = STRtree(geometries)
    for first_index, geometry in enumerate(geometries):
        for second_index in tree.query(geometry, predicate="intersects"):
            second_index = int(second_index)
            if second_index <= first_index:
                continue
            first = retained_features[first_index]["properties"]
            second = retained_features[second_index]["properties"]
            other = geometries[second_index]
            relation = (
                "equal_geometry"
                if geometry.equals(other)
                else "boundary_contact" if geometry.touches(other) else "area_overlap"
            )
            spatial_rows.append(
                {
                    "first_fragment": first["fragment_id"],
                    "second_fragment": second["fragment_id"],
                    "relation": relation,
                    "same_scoped_parcel_id": (
                        first["country"],
                        first["year"],
                        first["parcel_id"],
                    )
                    == (second["country"], second["year"], second["parcel_id"]),
                    "crop_conflict": first["crop"] != second["crop"],
                }
            )

output_path = field_folder / "download_candidates.geojson"
output_path.write_text(
    json.dumps({"type": "FeatureCollection", "features": retained_features}),
    encoding="utf-8",
)
(field_folder / "removed_exact_duplicates.json").write_text(
    json.dumps(duplicate_records, indent=2), encoding="utf-8"
)
(field_folder / "unresolved_field_ids.json").write_text(
    json.dumps(unresolved_ids, indent=2), encoding="utf-8"
)
with (field_folder / "remaining_spatial_review.csv").open(
    "w", newline="", encoding="utf-8"
) as report:
    writer = csv.DictWriter(
        report,
        fieldnames=[
            "first_fragment",
            "second_fragment",
            "relation",
            "same_scoped_parcel_id",
            "crop_conflict",
        ],
    )
    writer.writeheader()
    writer.writerows(spatial_rows)

deduplication_summary = {
    "original_fragments": len(original_features),
    "removed_exact_duplicates": len(duplicate_records),
    "download_candidates": len(retained_features),
    "unresolved_repeated_id_groups": len(unresolved_ids),
    "remaining_spatial_pairs": len(spatial_rows),
    "planned_24_bin_composites": len(retained_features) * 24,
    "policy": "Exact same-ID/crop/year/country footprints deduplicated; partial fragments and different-ID overlaps retained pending review.",
    "independent_field_count": "Not yet verified",
}
(field_folder / "deduplication_summary.json").write_text(
    json.dumps(deduplication_summary, indent=2), encoding="utf-8"
)
print(json.dumps(deduplication_summary, indent=2))
print("Step 5 will use:", output_path)


### 4.7 Select a balanced GeoTIFF pilot

The pilot uses 1,000 unique labeled pixels per crop. Prefer 20 geographically spread parcel groups per crop and 50 pixels per group. A fixed seed makes the choice reproducible.

A **partition group** combines country, year and parcel ID. It conservatively keeps possible field fragments together during later splitting; it is not a verified independent-field identity. Exclude pixels with conflicting crop or parcel-group labels.

Download rectangles around the selections. Matching masks mark the training pixels with 1; surrounding pixels remain available for visual inspection. Equal initial pixel budgets may become unequal after cloud and missing-data checks. Requiring enough pixels per group also excludes the smallest fragments from this pilot.


#### 4.7a Audit unique pixel locations

Recover native pixel centers and exclude conflicting labels/groups before sampling.


In [ ]:
import math
import json
import hashlib
from collections import defaultdict, Counter
import numpy as np
from affine import Affine
from rasterio.features import rasterize
from pyproj import Transformer
from shapely.geometry import shape, mapping, box
from shapely.ops import transform as transform_geometry, unary_union


def build_pixel_catalog(features):
    """Recover unique label-grid centers and retain all fragment provenance."""
    catalog = {}
    for feature in features:
        properties = feature["properties"]
        source_crs = properties["source_crs"]
        projector = Transformer.from_crs("EPSG:4326", source_crs, always_xy=True)
        geometry = transform_geometry(projector.transform, shape(feature["geometry"]))
        west, south, east, north = geometry.bounds
        # Tolerance prevents floating reprojection noise adding an extra row.
        left = math.floor(west / 10 + 1e-6) * 10
        right = math.ceil(east / 10 - 1e-6) * 10
        bottom = math.floor(south / 10 + 1e-6) * 10
        top = math.ceil(north / 10 - 1e-6) * 10
        width = int(round((right - left) / 10))
        height = int(round((top - bottom) / 10))
        if width < 1 or height < 1 or width * height > 5000000:
            raise ValueError(
                "Unexpected native pixel grid; investigate before sampling."
            )
        mask = rasterize(
            [(mapping(geometry), 1)],
            out_shape=(height, width),
            transform=Affine(10, 0, left, 0, -10, top),
            dtype="uint8",
        )
        rows, columns = np.where(mask == 1)
        group_key = (
            f"{properties['country']}|{properties['year']}|{properties['parcel_id']}"
        )
        for row, column in zip(rows, columns):
            x = int(round(left + int(column) * 10 + 5))
            y = int(round(top - int(row) * 10 - 5))
            key = (source_crs, x, y)
            record = catalog.setdefault(
                key,
                {
                    "source_crs": source_crs,
                    "x": x,
                    "y": y,
                    "crops": set(),
                    "groups": set(),
                    "source_fragments": set(),
                },
            )
            record["crops"].add(properties["crop"])
            record["groups"].add(group_key)
            record["source_fragments"].add(properties["fragment_id"])
    usable = []
    ambiguous = []
    for record in catalog.values():
        if len(record["crops"]) != 1 or len(record["groups"]) != 1:
            ambiguous.append(
                {
                    **record,
                    "crops": sorted(record["crops"]),
                    "groups": sorted(record["groups"]),
                    "source_fragments": sorted(record["source_fragments"]),
                }
            )
            continue
        usable.append(
            {
                "source_crs": record["source_crs"],
                "x": record["x"],
                "y": record["y"],
                "crop": next(iter(record["crops"])),
                "partition_group": next(iter(record["groups"])),
                "source_fragments": sorted(record["source_fragments"]),
            }
        )
    return (
        sorted(
            usable,
            key=lambda row: (
                row["crop"],
                row["partition_group"],
                row["source_crs"],
                row["x"],
                row["y"],
            ),
        ),
        ambiguous,
    )


def choose_spread_groups(groups, count, seed):
    """Use farthest-point selection for broad geographic coverage."""
    keys = sorted(groups)
    centers = []
    for key in keys:
        rows = groups[key]
        crs = rows[0]["source_crs"]
        if any(row["source_crs"] != crs for row in rows):
            raise ValueError(
                "One parcel group spans coordinate systems; review it first."
            )
        to_metric = Transformer.from_crs(crs, "EPSG:3857", always_xy=True)
        x, y = to_metric.transform(
            np.mean([row["x"] for row in rows]), np.mean([row["y"] for row in rows])
        )
        centers.append([x, y])
    centers = np.asarray(centers)
    rng = np.random.default_rng(seed)
    selected = [int(rng.integers(len(keys)))]
    distances = np.full(len(keys), np.inf)
    while len(selected) < count:
        newest = selected[-1]
        squared_distance = ((centers - centers[newest]) ** 2).sum(axis=1)
        distances = np.minimum(distances, squared_distance)
        distances[selected] = -1
        selected.append(int(np.argmax(distances)))
    return [keys[index] for index in selected]


candidate_path = field_folder / "download_candidates.geojson"
all_candidates = json.loads(candidate_path.read_text(encoding="utf-8"))["features"]
pixel_catalog, ambiguous_pixels = build_pixel_catalog(all_candidates)
print("Unique unambiguous pixels:", dict(Counter(row["crop"] for row in pixel_catalog)))
print("Ambiguous pixels excluded:", len(ambiguous_pixels))


#### 4.7b Set the pilot budget

Keep 1,000 pixels per crop, preferably 20 groups each, with a fixed seed and a shared group cap.


In [ ]:
# The user-selected pilot has equal initial pixel and location budgets.
pixels_per_crop = 1000
preferred_groups_per_crop = 20
max_pixels_per_group = 100
sampling_seed = 42
sampling_resolution = 10
sampling_buffer_metres = 20  # Must match Step 5.
pilot_folder = field_folder / "balanced_pilot"
pilot_folder.mkdir(parents=True, exist_ok=True)


#### 4.7c Select groups and pixels

Spread groups geographically, then draw pixels reproducibly within each group.


In [ ]:
import csv
import rasterio

# Build capacities using conservative parcel partition groups.
groups_by_crop = {crop: defaultdict(list) for crop in ["carrot", "potato"]}
for row in pixel_catalog:
    if row["crop"] in groups_by_crop:
        groups_by_crop[row["crop"]][row["partition_group"]].append(row)

# Use the same number of groups for both crops. Lower the group count only
# when too few groups can support the required per-group quotas.
chosen_group_count = None
for count in range(preferred_groups_per_crop, 0, -1):
    quota = math.ceil(pixels_per_crop / count)
    if quota > max_pixels_per_group:
        continue
    enough = all(
        sum(len(rows) >= quota for rows in groups.values()) >= count
        for groups in groups_by_crop.values()
    )
    if enough:
        chosen_group_count = count
        break
if chosen_group_count is None:
    raise ValueError(
        "Insufficient field-group capacity for this pilot; reduce the budget or revise the cap."
    )

selected_pixels = []
for crop_number, crop in enumerate(["carrot", "potato"]):
    quota = math.ceil(pixels_per_crop / chosen_group_count)
    eligible_groups = {
        key: rows for key, rows in groups_by_crop[crop].items() if len(rows) >= quota
    }
    chosen_groups = choose_spread_groups(
        eligible_groups, chosen_group_count, sampling_seed + crop_number
    )
    base_quota, remainder = divmod(pixels_per_crop, chosen_group_count)
    for group_number, key in enumerate(chosen_groups):
        group_rows = eligible_groups[key]
        take = base_quota + int(group_number < remainder)
        # Stable group-specific seed: no dependence on download/file order.
        group_seed = int(
            hashlib.sha256(f"{sampling_seed}|{crop}|{key}".encode()).hexdigest()[:16],
            16,
        )
        rng = np.random.default_rng(group_seed)
        indices = sorted(rng.choice(len(group_rows), size=take, replace=False))
        for index in indices:
            row = dict(group_rows[int(index)])
            row["pixel_id"] = hashlib.sha256(
                f"{row['source_crs']}|{row['x']}|{row['y']}".encode()
            ).hexdigest()[:20]
            selected_pixels.append(row)



#### 4.7d Build footprints and sampling masks

Create one retrieval region per selected group. Masks identify exactly the selected pixels on the Step 5 grid.


In [ ]:
# One download footprint and matching sampling mask per selected group.
selected_groups = defaultdict(list)
for row in selected_pixels:
    selected_groups[(row["crop"], row["partition_group"])].append(row)
pilot_regions = []
for (crop, group_key), rows in selected_groups.items():
    source_crs = rows[0]["source_crs"]
    pixel_shapes = [
        box(row["x"] - 5, row["y"] - 5, row["x"] + 5, row["y"] + 5) for row in rows
    ]
    projected_geometry = unary_union(pixel_shapes)
    to_wgs84 = Transformer.from_crs(source_crs, "EPSG:4326", always_xy=True)
    geographic_geometry = transform_geometry(to_wgs84.transform, projected_geometry)
    sample_id = hashlib.sha256(f"pilot|{crop}|{group_key}".encode()).hexdigest()[:20]
    properties = {
        "fragment_id": sample_id,
        "field_id": None,
        "partition_group": group_key,
        "crop": crop,
        "source_crs": source_crs,
        "selected_pixels": len(rows),
        "source_fragments": sorted(
            set(fragment for row in rows for fragment in row["source_fragments"])
        ),
        "geometry_status": "selected_pixel_footprints",
        "sampling_seed": sampling_seed,
    }
    region = {
        "type": "Feature",
        "properties": properties,
        "geometry": mapping(geographic_geometry),
    }

    # Match Step 5's outward-rounded download grid exactly, including its buffer.
    back_projector = Transformer.from_crs("EPSG:4326", source_crs, always_xy=True)
    roundtrip_geometry = transform_geometry(
        back_projector.transform, geographic_geometry
    )
    west, south, east, north = roundtrip_geometry.bounds
    left = (
        math.floor((west - sampling_buffer_metres) / sampling_resolution)
        * sampling_resolution
    )
    right = (
        math.ceil((east + sampling_buffer_metres) / sampling_resolution)
        * sampling_resolution
    )
    bottom = (
        math.floor((south - sampling_buffer_metres) / sampling_resolution)
        * sampling_resolution
    )
    top = (
        math.ceil((north + sampling_buffer_metres) / sampling_resolution)
        * sampling_resolution
    )
    width = int(round((right - left) / sampling_resolution))
    height = int(round((top - bottom) / sampling_resolution))
    if width * height * 7 * 4 >= 30 * 1024**2:
        raise ValueError(
            "Sample group has a large/disconnected footprint; review before downloading."
        )
    sample_mask = np.zeros((height, width), dtype="uint8")
    for row in rows:
        column = int(math.floor((row["x"] - left) / sampling_resolution))
        raster_row = int(math.floor((top - row["y"]) / sampling_resolution))
        sample_mask[raster_row, column] = 1
        row["download_region_id"] = sample_id
    if int(sample_mask.sum()) != len(rows):
        raise ValueError("Selected centers do not map uniquely onto the planned grid.")
    mask_path = pilot_folder / "sampling_masks" / f"{sample_id}.tif"
    mask_path.parent.mkdir(exist_ok=True)
    with rasterio.open(
        mask_path,
        "w",
        driver="GTiff",
        width=width,
        height=height,
        count=1,
        dtype="uint8",
        crs=source_crs,
        transform=Affine(sampling_resolution, 0, left, 0, -sampling_resolution, top),
        compress="deflate",
    ) as raster:
        raster.write(sample_mask, 1)
        raster.set_band_description(1, "selected_training_pixel")
    properties["sampling_mask"] = str(mask_path.resolve())
    properties["sampling_grid"] = {
        "crs": source_crs,
        "transform": [sampling_resolution, 0, left, 0, -sampling_resolution, top],
        "width": width,
        "height": height,
    }
    pilot_regions.append(region)



#### 4.7e Save the pilot records

Write selected coordinates, region footprints, exclusions and counts. Original candidates remain unchanged.


In [ ]:
region_path = field_folder / "sampled_download_regions.geojson"
region_path.write_text(
    json.dumps({"type": "FeatureCollection", "features": pilot_regions}),
    encoding="utf-8",
)
columns = [
    "pixel_id",
    "crop",
    "partition_group",
    "download_region_id",
    "source_crs",
    "x",
    "y",
    "source_fragments",
]
with (pilot_folder / "selected_pixels.csv").open(
    "w", newline="", encoding="utf-8"
) as report:
    writer = csv.DictWriter(report, fieldnames=columns)
    writer.writeheader()
    for row in selected_pixels:
        writer.writerow({**row, "source_fragments": ";".join(row["source_fragments"])})
(pilot_folder / "ambiguous_pixels.json").write_text(
    json.dumps(ambiguous_pixels, indent=2), encoding="utf-8"
)
sampling_summary = {
    "pixel_counts": dict(Counter(row["crop"] for row in selected_pixels)),
    "groups_per_crop": chosen_group_count,
    "download_regions": len(pilot_regions),
    "composite_files": len(pilot_regions) * 24,
    "sampling_seed": sampling_seed,
    "ambiguous_pixels_excluded": len(ambiguous_pixels),
    "maximum_pixels_per_group": max(len(rows) for rows in selected_groups.values()),
    "scope": "Balanced pilot, not full crop inventory; eligibility favors groups able to supply the quota.",
    "field_count_status": "Partition groups are conservative parcel-ID groups, not verified independent fields.",
    "quality_status": "Initial labels only; cloud completeness and crop-cycle relevance still need validation.",
}
(pilot_folder / "sampling_summary.json").write_text(
    json.dumps(sampling_summary, indent=2), encoding="utf-8"
)
print(json.dumps(sampling_summary, indent=2))
print("Step 5 retrieval list:", region_path)


### Outputs and interpretation

`candidate_fragments.geojson` preserves original fragments. `download_candidates.geojson` removes verified redundant footprints. Review tables record unresolved IDs, overlaps and errors. The pilot adds `sampled_download_regions.geojson`, selected-pixel CSV records and 40 mask GeoTIFFs. Shared field identities and crop validity still need verification.


## 5. Download cloud-screened median composites

Use the pilot's selected-pixel regions, not the full fragment inventory. Each region has 24 calendar half-month intervals in 2020: days 1–15 and days 16–month-end. The prepared 40-region pilot therefore has 960 planned GeoTIFFs.

Apply **SCL and cloud probability before calculating medians**. Exclude SCL invalid pixels, clouds, shadows, cirrus and snow. Also exclude cloud scores at or above 40 and invalid scene edges. Missing cloud-probability matches are reported and excluded. The score cutoff is a development choice, not a percentage of the whole scene covered by clouds. Shadow screening uses SCL; no separate projected-shadow detector is implemented.

First mask each granule. Combine valid granules from the same date in sorted image-index order; the last valid granule wins an overlap. Then calculate the median across distinct dates. This gives each date equal weight. Do not confuse this deterministic overlap rule with cloud-score ranking.

GeoTIFF band order is B3, B4, B5, B8, B11, B2 and `clear_count`. Spectral medians remain unscaled float32 values; missing spectral values are −9999. `clear_count` records supporting dates at each pixel. Empty intervals remain present with count zero. The 10 m grid does not add detail to bands originally measured at 20 m.

Old acquisition downloads remain untouched. Mask and grid settings determine the request folder, so changed settings do not silently reuse incompatible composites. Downloads have no per-run cap and save progress after each file.

References: [L2A collection](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED), [cloud probability](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_CLOUD_PROBABILITY), and [direct-download limits](https://developers.google.com/earth-engine/apidocs/ee-image-getdownloadurl).


### 5.1 Install retrieval packages — once

Skip when Earth Engine and requests are already installed.


In [ ]:
%pip install earthengine-api requests


### 5.2 Set retrieval and cloud settings

Load pilot regions, dates, output paths and the cloud-score cutoff. This cell does not download imagery.


In [ ]:
import os
import json
from pathlib import Path

# Collaborators can set environment variables or edit these two settings.
# dataset_folder was defined in Step 2; its sibling folder holds large imagery.
l2a_folder = Path(
    os.environ.get(
        "L2A_COMPOSITE_FOLDER",
        local_settings.get(
            "imagery_root", str(dataset_folder / "sentinel2_pilot_composites")
        ),
    )
).resolve()
earth_engine_project = os.environ.get(
    "EE_PROJECT",
    local_settings.get("earth_engine_project", "YOUR_EARTH_ENGINE_PROJECT"),
)

collection_name = "COPERNICUS/S2_SR_HARMONIZED"
start_date = "2020-01-01"  # Inclusive.
end_date = "2021-01-01"  # Exclusive; extend if adjacent months are needed.
output_resolution = 10  # Metres; confirmed study grid.
buffer_metres = 20  # Rectangular context around each candidate fragment.
spectral_bands = ["B3", "B4", "B5", "B8", "B11", "B2"]
output_bands = spectral_bands + ["clear_count"]
excluded_scl_classes = [0, 1, 3, 8, 9, 10, 11]
cloud_probability_collection = "COPERNICUS/S2_CLOUD_PROBABILITY"
cloud_probability_threshold = 40  # Exclude scores >= 40; exploratory pilot setting.
nodata_value = -9999  # Used only for missing spectral values; clear_count remains zero.
downloads_per_run = None  # Also set explicitly in the download cell.
authenticate_now = False
refresh_inventory = False

from datetime import date

if date.fromisoformat(start_date) >= date.fromisoformat(end_date):
    raise ValueError("end_date must be later than start_date.")
if output_resolution != 10 or buffer_metres < 0:
    raise ValueError("This implementation uses a 10 m grid and nonnegative buffer.")
if downloads_per_run is not None and downloads_per_run < 1:
    raise ValueError("downloads_per_run must be positive or None.")

fragment_path = (
    report_folder.parent / "fields" / "2020" / "sampled_download_regions.geojson"
)
if not fragment_path.exists():
    raise FileNotFoundError(
        "Run Step 4 balanced-pilot preparation before creating a Step 5 plan."
    )
fragments = json.loads(fragment_path.read_text(encoding="utf-8"))["features"]
if not fragments:
    raise ValueError("Step 4 produced no candidate fragments.")
l2a_reports = report_folder.parent / "l2a" / "2020"
l2a_reports.mkdir(parents=True, exist_ok=True)
print("Candidate fragments:", len(fragments))
print("Imagery storage:", l2a_folder)
print("Earth Engine project:", earth_engine_project)
print("Observation period:", start_date, "to", end_date, "(end exclusive)")

if start_date != "2020-01-01" or end_date != "2021-01-01":
    raise ValueError("This composite plan currently covers full 2020 only.")

if not 0 < cloud_probability_threshold <= 100:
    raise ValueError(
        "Cloud probability threshold must be between 0 (exclusive) and 100."
    )


### 5.3 Connect to Earth Engine

Use the registered project. Set authenticate_now only when sign-in is needed; never store tokens in the notebook.


In [ ]:
import ee

# Sign in only when needed. Credentials are managed by Earth Engine,
# not written into the shared notebook.
if authenticate_now:
    ee.Authenticate()

try:
    ee.Initialize(project=earth_engine_project)
    print("Connected to Earth Engine.")
except Exception as error:
    raise RuntimeError(
        "Earth Engine connection failed. Check the project registration and access. "
        "If sign-in is needed, set authenticate_now = True and rerun settings and this cell."
    ) from error


### 5.4a Define grids and request settings

Build fixed grids and fingerprints so incompatible outputs are not reused.


In [ ]:
import math
import hashlib
import csv
import requests
import rasterio
from affine import Affine
from pyproj import CRS, Transformer
from shapely.geometry import shape
from shapely.ops import transform as transform_geometry


def save_json(path, data):
    """Replace a report only after its complete content is written."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(json.dumps(data, indent=2), encoding="utf-8")
    temporary_path.replace(path)


def make_download_grid(fragment):
    """Create a fixed rectangular grid in the fragment's verified source CRS."""
    source_crs = CRS.from_user_input(fragment["properties"]["source_crs"])
    if not source_crs.is_projected or any(
        axis.unit_name.lower() not in ["metre", "meter"]
        for axis in source_crs.axis_info
    ):
        raise ValueError(
            "The download grid requires a verified projected CRS in metres."
        )
    geometry = shape(fragment["geometry"])
    if geometry.is_empty or not geometry.is_valid:
        raise ValueError("Invalid candidate geometry.")
    projector = Transformer.from_crs("EPSG:4326", source_crs, always_xy=True)
    projected = transform_geometry(projector.transform, geometry)
    west, south, east, north = projected.bounds
    # Round outward on a common 10 m lattice, so acquisitions share a grid.
    left = math.floor((west - buffer_metres) / output_resolution) * output_resolution
    right = math.ceil((east + buffer_metres) / output_resolution) * output_resolution
    bottom = math.floor((south - buffer_metres) / output_resolution) * output_resolution
    top = math.ceil((north + buffer_metres) / output_resolution) * output_resolution
    width = int(round((right - left) / output_resolution))
    height = int(round((top - bottom) / output_resolution))
    return {
        "crs": source_crs.to_string(),
        "transform": [output_resolution, 0, left, 0, -output_resolution, top],
        "width": width,
        "height": height,
    }


def request_settings(fragment):
    """Settings fingerprint prevents reuse of files from a different request."""
    return {
        "collection": collection_name,
        "start": start_date,
        "end": end_date,
        "bands": output_bands,
        "nodata": nodata_value,
        "grid": make_download_grid(fragment),
        "geometry": fragment["geometry"],
        "resampling": "nearest",
        "partition_group": fragment["properties"].get("partition_group"),
        "sampling_mask": fragment["properties"].get("sampling_mask"),
        "reflectance_scale_to_apply_later": 0.0001,
        "version": 3,
        "compositing": "median of cloud-screened daily mosaics in calendar half-month bins",
        "excluded_scl_classes": excluded_scl_classes,
        "cloud_probability_collection": cloud_probability_collection,
        "cloud_probability_threshold": cloud_probability_threshold,
        "missing_cloud_probability": "exclude acquisition, report missing match",
        "scene_edge_masks": ["B8A", "B9"],
    }




### 5.4b Define file checks and inventory helpers

Check raster properties/checksums and request original acquisition metadata.


In [ ]:
def checksum(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def check_raster(path, grid):
    """Check dimensions, CRS, transform, band count and sample readability."""
    with rasterio.open(path) as raster:
        if raster.count != len(output_bands):
            raise ValueError("Unexpected downloaded band count.")
        if (raster.width, raster.height) != (grid["width"], grid["height"]):
            raise ValueError("Downloaded raster dimensions do not match the plan.")
        if CRS.from_user_input(raster.crs) != CRS.from_user_input(grid["crs"]):
            raise ValueError("Downloaded CRS does not match the plan.")
        if not raster.transform.almost_equals(Affine(*grid["transform"])):
            raise ValueError("Downloaded raster transform does not match the plan.")
        if any(dtype != "float32" for dtype in raster.dtypes):
            raise ValueError("Expected float32 medians and clear-observation counts.")
        # Check that pixel data can be decoded, not just the header.
        raster.read(
            window=rasterio.windows.Window(
                0, 0, min(16, raster.width), min(16, raster.height)
            )
        )


def list_acquisitions(fragment):
    """Fetch only small metadata for original granules intersecting a fragment."""
    footprint = ee.Geometry(fragment["geometry"], proj="EPSG:4326", geodesic=False)
    collection = (
        ee.ImageCollection(collection_name)
        .filterBounds(footprint)
        .filterDate(start_date, end_date)
    )

    def describe(image):
        image = ee.Image(image)
        return ee.Feature(
            None,
            {
                "image_index": image.get("system:index"),
                "timestamp_ms": image.get("system:time_start"),
                "date": ee.Date(image.get("system:time_start")).format("YYYY-MM-dd"),
                "tile": image.get("MGRS_TILE"),
                "scene_cloud_percent": image.get("CLOUDY_PIXEL_PERCENTAGE"),
            },
        )

    metadata = ee.FeatureCollection(
        collection.toList(collection.size()).map(describe)
    ).getInfo()
    rows = []
    for feature in metadata["features"]:
        row = feature["properties"]
        row["image_id"] = collection_name + "/" + row["image_index"]
        rows.append(row)
    return sorted(rows, key=lambda row: (row["timestamp_ms"], row["image_index"]))




### 5.4c Define masking and compositing

Create calendar intervals, match cloud images, mask acquisitions and calculate daily-mosaic medians.


In [ ]:
def make_half_month_bins():
    """Generate 24 start-inclusive, end-exclusive calendar intervals."""
    from datetime import date

    bins = []
    for month in range(1, 13):
        first = date(2020, month, 1)
        middle = date(2020, month, 16)
        following = date(2020, month + 1, 1) if month < 12 else date(2021, 1, 1)
        bins.append(
            {
                "name": f"2020_{month:02d}_01_15",
                "start": first.isoformat(),
                "end": middle.isoformat(),
            }
        )
        bins.append(
            {
                "name": f"2020_{month:02d}_16_end",
                "start": middle.isoformat(),
                "end": following.isoformat(),
            }
        )
    return bins


def find_cloud_probability_matches(fragment):
    """Match original granules by exact system:index, not nearest date."""
    footprint = ee.Geometry(fragment["geometry"], proj="EPSG:4326", geodesic=False)
    clouds = (
        ee.ImageCollection(cloud_probability_collection)
        .filterBounds(footprint)
        .filterDate(start_date, end_date)
    )
    return set(clouds.aggregate_array("system:index").getInfo())


def mask_acquisition(image, cloud_probability_image):
    """Combine SCL, cloud probability and valid-data masks BEFORE medians."""
    image = ee.Image(image)
    scl = image.select("SCL")
    excluded = scl.remap(excluded_scl_classes, [1] * len(excluded_scl_classes), 0)
    probability = ee.Image(cloud_probability_image).select("probability")
    low_cloud_score = probability.lt(cloud_probability_threshold)
    reflectance = image.select(spectral_bands)
    all_bands_valid = reflectance.mask().reduce(ee.Reducer.min())
    # Additional edge masks follow the Earth Engine cloud-probability example.
    edges_valid = image.select("B8A").mask().And(image.select("B9").mask())
    valid = excluded.eq(0).And(scl.mask()).And(all_bands_valid)
    valid = valid.And(low_cloud_score).And(probability.mask()).And(edges_valid)
    return reflectance.updateMask(valid).copyProperties(
        image, ["system:time_start", "system:index"]
    )


def make_composite(fragment, interval, granules):
    """Use at most one observation per date, then take the semi-monthly median."""
    matched_granules = [row for row in granules if row.get("cloud_probability_id")]
    dates = sorted(set(row["date"] for row in matched_granules))
    daily_images = []
    for observation_date in dates:
        same_date = sorted(
            [row for row in matched_granules if row["date"] == observation_date],
            key=lambda row: row["image_index"],
        )
        images = [
            mask_acquisition(
                ee.Image(row["image_id"]), ee.Image(row["cloud_probability_id"])
            )
            for row in same_date
        ]
        # Mask each granule first. Later sorted granules win at valid overlaps.
        daily = ee.ImageCollection.fromImages(images).mosaic()
        daily_images.append(daily)
    if daily_images:
        daily_collection = ee.ImageCollection.fromImages(daily_images)
        median = daily_collection.median().select(spectral_bands)
        clear_count = (
            daily_collection.select("B3")
            .count()
            .rename("clear_count")
            .unmask(0, sameFootprint=False)
        )
        # Keep count=0 even where all spectral medians are missing.
        result = median.unmask(nodata_value, sameFootprint=False).addBands(clear_count)
    else:
        # Do not omit empty bins or pretend they have observed values.
        result = ee.Image.constant([nodata_value] * len(spectral_bands) + [0]).rename(
            output_bands
        )
    return result.toFloat()


### 5.5 Build the composite plan

Reuse/query acquisition metadata for all pilot regions. Report missing exact cloud matches; no imagery is downloaded here.


In [ ]:
# Inventory selected pilot regions only, then create their 24-bin composite plan.
# No need to query every field's acquisitions again after the previous inventory.
acquisition_plans = []
inventory_errors = []
existing_inventory = l2a_reports / "acquisition_inventory"
for number, fragment in enumerate(fragments, start=1):
    fragment_id = fragment["properties"]["fragment_id"]
    saved_path = existing_inventory / f"{fragment_id}.json"
    try:
        saved = (
            json.loads(saved_path.read_text(encoding="utf-8"))
            if saved_path.exists()
            else None
        )
        can_reuse = (
            saved
            and saved["settings"]["collection"] == collection_name
            and saved["settings"]["start"] == start_date
            and saved["settings"]["end"] == end_date
            and saved["settings"]["geometry"] == fragment["geometry"]
        )
        if can_reuse and not refresh_inventory:
            acquisitions = saved["acquisitions"]
        else:
            acquisitions = list_acquisitions(fragment)
            save_json(
                saved_path,
                {
                    "fragment_id": fragment_id,
                    "settings": request_settings(fragment),
                    "properties": fragment["properties"],
                    "acquisitions": acquisitions,
                },
            )
        # Reuse cloud matches only when this inventory explicitly records them.
        cloud_matches_cached = (
            can_reuse
            and not refresh_inventory
            and saved.get("cloud_probability_collection")
            == cloud_probability_collection
        )
        if cloud_matches_cached:
            acquisitions = saved["acquisitions"]
        else:
            matching_indices = find_cloud_probability_matches(fragment)
            for acquisition in acquisitions:
                index = acquisition["image_index"]
                acquisition["cloud_probability_id"] = (
                    cloud_probability_collection + "/" + index
                    if index in matching_indices
                    else None
                )
            save_json(
                saved_path,
                {
                    "fragment_id": fragment_id,
                    "settings": request_settings(fragment),
                    "properties": fragment["properties"],
                    "acquisitions": acquisitions,
                    "cloud_probability_collection": cloud_probability_collection,
                },
            )
        acquisition_plans.append(
            {
                "fragment_id": fragment_id,
                "fragment": fragment,
                "settings": request_settings(fragment),
                "acquisitions": acquisitions,
            }
        )
    except Exception as error:
        inventory_errors.append({"fragment_id": fragment_id, "error": str(error)})

half_month_bins = make_half_month_bins()
save_json(
    l2a_reports / "composite_plan.json",
    {
        "fragments": len(acquisition_plans),
        "intervals": half_month_bins,
        "planned_composites": len(acquisition_plans) * len(half_month_bins),
        "failed_inventory": inventory_errors,
        "bands": output_bands,
        "mask": "SCL plus cloud probability plus edge-validity screening before daily mosaics and medians",
        "cloud_probability_collection": cloud_probability_collection,
        "cloud_probability_threshold": cloud_probability_threshold,
        "granules_without_cloud_match": sum(
            not row.get("cloud_probability_id")
            for plan in acquisition_plans
            for row in plan["acquisitions"]
        ),
        "mosaic_rule": "Per date, sorted system:index; later valid granules take priority. Different dates have equal weight.",
        "clear_count": "Number of unique dates with all six spectral bands valid at the pixel.",
    },
)
print("Fragments:", len(acquisition_plans))
print("Composites per fragment:", len(half_month_bins))
print("Total planned composite files:", len(acquisition_plans) * len(half_month_bins))
print("Inventory errors:", len(inventory_errors))

missing_matches = [
    {
        "fragment_id": plan["fragment_id"],
        "image_id": row["image_id"],
        "reason": "No exact cloud probability match; excluded from compositing",
    }
    for plan in acquisition_plans
    for row in plan["acquisitions"]
    if not row.get("cloud_probability_id")
]
save_json(l2a_reports / "missing_cloud_probability.json", missing_matches)
print("Granules excluded for missing cloud match:", len(missing_matches))


### 5.6 Download unfinished composites

This cell can run for a long time. Reuse validated files and checkpoint each new attempt. There is no per-run cap.


In [ ]:
# Download 24 semi-monthly medians per fragment, not every acquisition.
# Earth Engine computes the masks, daily mosaics and medians before download.
downloads_per_run = None
print("Download mode: all unfinished semi-monthly composites.")
network_attempts = reused_files = downloaded_files = 0
download_errors = []
limit_reached = False

for plan in acquisition_plans:
    fragment_id = plan["fragment_id"]
    retrieval_settings = plan["settings"]
    grid = retrieval_settings["grid"]
    expected_grid = plan["fragment"]["properties"].get("sampling_grid")
    if expected_grid and grid != expected_grid:
        raise ValueError(
            "Sampling mask grid differs from the download grid; regenerate the pilot with matching settings."
        )
    estimated_bytes = grid["width"] * grid["height"] * len(output_bands) * 4
    if (
        max(grid["width"], grid["height"]) >= 10000
        or estimated_bytes >= 30 * 1024 * 1024
    ):
        download_errors.append(
            {
                "fragment_id": fragment_id,
                "error": "Composite needs tiled/export retrieval; direct-request limit exceeded.",
            }
        )
        continue
    request_id = hashlib.sha256(
        json.dumps(retrieval_settings, sort_keys=True).encode()
    ).hexdigest()[:16]
    destination_folder = l2a_folder / fragment_id / request_id
    destination_folder.mkdir(parents=True, exist_ok=True)
    manifest_path = (
        l2a_reports / "composite_progress" / f"{fragment_id}_{request_id}.json"
    )
    manifest = (
        json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest_path.exists()
        else {"fragment_id": fragment_id, "settings": retrieval_settings, "files": {}}
    )

    for interval in half_month_bins:
        name = interval["name"]
        destination = destination_folder / f"{name}.tif"
        granules = [
            row
            for row in plan["acquisitions"]
            if interval["start"] <= row["date"] < interval["end"]
        ]
        previous = manifest["files"].get(name)
        reusable = False
        if previous and previous.get("status") == "complete" and destination.is_file():
            try:
                check_raster(destination, grid)
                reusable = checksum(destination) == previous["sha256"]
            except Exception:
                pass
        if reusable:
            reused_files += 1
            continue
        if downloads_per_run is not None and network_attempts >= downloads_per_run:
            limit_reached = True
            break
        network_attempts += 1
        record = {
            **interval,
            "path": str(destination),
            "status": "pending",
            "source_image_ids": [row["image_id"] for row in granules],
            "available_dates": sorted(set(row["date"] for row in granules)),
        }
        record["cloud_screened_available_dates"] = sorted(
            set(row["date"] for row in granules if row.get("cloud_probability_id"))
        )
        record["excluded_missing_cloud_match"] = [
            row["image_id"] for row in granules if not row.get("cloud_probability_id")
        ]
        temporary_path = destination.with_suffix(".tif.part")
        try:
            composite = make_composite(plan["fragment"], interval, granules)
            url = composite.getDownloadURL(
                {
                    "name": name,
                    "format": "GEO_TIFF",
                    "filePerBand": False,
                    "crs": grid["crs"],
                    "crs_transform": grid["transform"],
                    "dimensions": [grid["width"], grid["height"]],
                }
            )
            with requests.get(url, stream=True, timeout=(30, 180)) as response:
                response.raise_for_status()
                with temporary_path.open("wb") as stream:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            stream.write(chunk)
            check_raster(temporary_path, grid)
            with rasterio.open(temporary_path, "r+") as raster:
                raster.nodata = nodata_value
                for number, band in enumerate(output_bands, start=1):
                    raster.set_band_description(number, band)
                counts = raster.read(len(output_bands))
                record["pixels_with_observations"] = int((counts > 0).sum())
                record["maximum_clear_dates"] = int(counts.max())
                record["empty_after_screening"] = not bool((counts > 0).any())
            record.update(
                status="complete",
                sha256=checksum(temporary_path),
                bytes=temporary_path.stat().st_size,
            )
            temporary_path.replace(destination)
            downloaded_files += 1
            print(
                "Composite:",
                fragment_id,
                name,
                "supported pixels:",
                record["pixels_with_observations"],
            )
        except Exception as error:
            record.update(status="failed", error=f"{type(error).__name__}: {error}")
            download_errors.append(
                {"fragment_id": fragment_id, "interval": name, "error": record["error"]}
            )
            print("Composite failed:", fragment_id, name, str(error))
        manifest["files"][name] = record
        save_json(manifest_path, manifest)
    if limit_reached:
        break



### 5.7 Summarize this download run

Record the counters and errors from the previous cell. A run summary alone does not prove full-plan completion.


In [ ]:
save_json(
    l2a_reports / "last_composite_run.json",
    {
        "downloaded": downloaded_files,
        "reused": reused_files,
        "attempts": network_attempts,
        "errors": download_errors,
        "limit_reached": limit_reached,
    },
)
print(
    "New composites:",
    downloaded_files,
    "Reused:",
    reused_files,
    "Errors:",
    len(download_errors),
)
print("Composite storage:", l2a_folder)


### Outputs and interpretation

`composite_plan.json` records intervals and screening rules. `composite_progress/` stores paths, source IDs, support counts and checksums; `missing_cloud_probability.json` records excluded matches. Imagery lives in the configured pilot folder. Only pixels marked 1 in sampling masks belong to the pilot; the extra rectangle pixels are context.


## 6. Check quality, scaling and alignment

Audit every interval, including missing or failed downloads. Verify the combined-cloud-mask version, band order, sampling-mask grid and clear-observation counts. Scale spectral values by 0.0001 after handling nodata; never scale counts.

The initial coverage rule requires valid values at 80% of selected pixels. This is an exploratory setting to inspect before freezing. Preserve low-coverage and invalid records instead of filling missing values.

Original GeoTIFFs stay unchanged. On another PC, the helpers try current project and imagery roots before old absolute paths. Cloud-contaminated medians require upstream regeneration; quality checks cannot repair them afterwards.


### 6.1 Audit all pilot intervals

Run local quality checks using the previous settings and existing GeoTIFFs. Missing intervals remain in the report.


In [ ]:
from pathlib import Path
import importlib
import crop_workflow

importlib.reload(crop_workflow)

field_folder = report_folder.parent / "fields" / "2020"
regions = json.loads(
    (field_folder / "sampled_download_regions.geojson").read_text(encoding="utf-8")
)["features"]
analysis_folder = report_folder.parent / "analysis" / "2020"
minimum_valid_fraction = 0.8  # Development setting; inspect exclusions before freezing.

quality_rows, pixel_rows = crop_workflow.quality_check(
    regions=regions,
    progress_folder=l2a_reports / "composite_progress",
    imagery_folder=l2a_folder,
    field_folder=field_folder,
    output_folder=analysis_folder / "quality",
    cloud_threshold=cloud_probability_threshold,
    minimum_fraction=minimum_valid_fraction,
)
from collections import Counter

print("Quality status:", dict(Counter(row["status"] for row in quality_rows)))
print("Reports:", analysis_folder / "quality")


### Outputs and interpretation

`analysis/2020/quality/` contains quality records, scaled selected-pixel bands, settings and errors. Missing and low-coverage intervals remain explicit.


## 7. Build vegetation timelines

Calculate NDVI, NDRE and NDMI from each selected pixel's composite bands. Then summarize the selected pixels in each region. Composite-bin midpoints are display dates, not actual acquisition dates.

For stronger onset evidence, retrieve small daily index summaries using the same cloud mask, sampling footprint and same-date mosaic rule. This transfers tables only, not additional image files. Checkpoints let successful regions resume without repeating retrieval.

These are selected-pixel summaries, not full-field averages. A rise in an index does not by itself establish planting or crop identity.


### 7.1 Calculate composite curves and plots

Use the scaled pixel bands and quality records from Step 6. The first plot is displayed as an example.


In [ ]:
timeline_folder = analysis_folder / "timelines"
composite_rows = crop_workflow.composite_timelines(
    quality_rows, pixel_rows, timeline_folder
)
crop_workflow.plot_timelines(
    composite_rows,
    timeline_folder / "composite_plots",
    title="Semi-monthly composite summaries (bin midpoints)",
)
print("Composite timeline rows:", len(composite_rows))

# Optional visual preview of one saved plot.
from IPython.display import Image, display

plot_files = sorted((timeline_folder / "composite_plots").glob("*.png"))
if plot_files:
    display(Image(filename=str(plot_files[0])))


### 7.2 Retrieve observed daily summaries

This cell needs the Step 5 connection, helpers and acquisition plan. Skip it only when using saved observed summaries or explicitly accepting provisional composite-only detection.


In [ ]:
# Fetch small actual-date summaries; requires the Step 5 Earth Engine connection.
import observed_timelines

importlib.reload(observed_timelines)
observed_rows = observed_timelines.retrieve_observed_timelines(
    plans=acquisition_plans,
    mask_acquisition=mask_acquisition,
    ee=ee,
    destination=timeline_folder,
    minimum_fraction=minimum_valid_fraction,
)
if observed_rows:
    crop_workflow.plot_timelines(
        observed_rows,
        timeline_folder / "observed_plots",
        title="Observed daily selected-pixel summaries",
    )
print("Observed summary rows:", len(observed_rows))


### Outputs and interpretation

`analysis/2020/timelines/` contains composite curves, pixel indices and plots. Daily retrieval adds observed JSON/CSV summaries, checkpoints, errors and plots. Neither curve type is a verified planting record.


## 8. Identify candidate green-up intervals

The detector looks for a supported rise after a low-vegetation period. It uses actual daily summaries where available and falls back to composites for other regions. Composite-only candidates remain provisional.

| Development setting | Default |
|---|---:|
| Relative rise above baseline | 20% of the estimated amplitude |
| Minimum NDVI amplitude | 0.15 |
| Maximum gap around onset | 30 days |
| Later confirmations | 2 within 30 days |
| Baseline lookback | 120 days |
| Minimum valid sample fraction | 80% |

These are proposed settings, not validated vegetable constants. Baseline uses a local 10th percentile before each candidate peak. Short-neighbour median smoothing does not interpolate long gaps. Raw observations must also support the rise and confirmation.

Report an uncertainty interval rather than an exact planting date. Flag weak signals, gaps, multiple possible cycles and intervals crossing month boundaries. Step 9 must validate the detector and annual-label relevance before accepting any training season. Cycles near year boundaries may require additional months of observations.


### 8.1 Run the candidate detector

Prefer observed rows and preserve composite-only fallbacks. Save outcomes and the exact development settings.


In [ ]:
detector_settings = dict(crop_workflow.DEFAULT_DETECTOR)
# Edit these visible development settings before running sensitivity checks.
detector_settings["relative_threshold"] = 0.20

observed_path = timeline_folder / "observed_timelines.json"
observed_rows = (
    json.loads(observed_path.read_text(encoding="utf-8"))
    if observed_path.exists()
    else []
)
observed_regions = {row["region_id"] for row in observed_rows}
detector_rows = observed_rows + [
    row for row in composite_rows if row["region_id"] not in observed_regions
]
greenup_results = crop_workflow.detect_greenup(detector_rows, detector_settings)
greenup_folder = analysis_folder / "greenup"
crop_workflow.save_json(greenup_folder / "candidate_greenup.json", greenup_results)
crop_workflow.save_json(
    greenup_folder / "detector_settings.json",
    {
        "parameters": detector_settings,
        "validated": False,
        "label_relevance_verified": False,
    },
)
columns = [
    "region_id",
    "partition_group",
    "status",
    "reason",
    "source_kind",
    "onset_interval_start",
    "onset_interval_end",
    "peak_date",
    "baseline",
    "amplitude",
    "ndvi_threshold",
    "gap_days",
    "confirmations",
    "month_boundary_uncertain",
    "label_relevance",
    "validated",
]
crop_workflow.write_table(
    greenup_folder / "candidate_greenup.csv", greenup_results, columns
)
print("Detector outcomes:", dict(Counter(row["status"] for row in greenup_results)))
print("Results:", greenup_folder)



### 8.2 Compare development thresholds

Repeat detection at 15%, 20% and 25% relative amplitude without using final test labels.


In [ ]:
# Compare development thresholds without using final test labels.
sensitivity = {}
for fraction in [0.15, 0.20, 0.25]:
    results = crop_workflow.detect_greenup(
        detector_rows, {**detector_settings, "relative_threshold": fraction}
    )
    sensitivity[str(fraction)] = results
crop_workflow.save_json(greenup_folder / "threshold_sensitivity.json", sensitivity)



### 8.3 Plot candidate uncertainty intervals

Plot the primary detector output. Shaded intervals are estimates, not verified planting dates.


In [ ]:
crop_workflow.plot_greenup(detector_rows, greenup_results, greenup_folder / "plots")


### Outputs and interpretation

`analysis/2020/greenup/` contains candidate intervals, detector settings, sensitivity results and plots. Candidate, provisional, ambiguous and rejected outcomes remain separate. No season is automatically accepted for training.


## 9. Validate growth estimates and label relevance

**Work to implement**

Compare estimates with independent emergence/green-up evidence where available. Planting is not green-up; decline is not necessarily harvest. Annual crop labels must be linked to the selected cycle. Assess gaps, weak signals, multiple cycles, threshold sensitivity and region-specific failures. Calibrate on development fields and freeze rules before final testing. Without independent dates, report estimates or review agreement rather than verified planting accuracy.

**Expected output**

Frozen detector settings, uncertainty records and accepted field-season labels.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


### Planned outputs

Expected outputs are validation records, frozen detector settings, onset uncertainties and accepted or rejected field-season crop labels with reasons. These establish which cycles have adequate evidence for later window construction. Without independent timing evidence, results remain estimates rather than verified planting accuracy. Implementation is not included yet.


## 10. Select four consecutive calendar months

**Work to implement**

Proposed rule: start on the first day of the month containing accepted green-up, then take that month and the next three. Flag onset intervals crossing month boundaries. Check that the window corresponds to the declared crop and does not contain an unresolved rotation. Do not compress seasons or concatenate separated months. The same detector must work on unknown Benguet crops; field-level detection requires boundaries. Peak-based detection is retrospective and may need imagery beyond the classifier window.

**Expected output**

Accepted window start/end, selection evidence and exclusion log.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


### Planned outputs

Expected outputs are accepted window start/end dates, green-up evidence, month-boundary uncertainty flags and an exclusion log. Each accepted record identifies four consecutive calendar months relevant to the declared crop. These dates define the eight bins in Step 11. Implementation is not included yet.


## 11. Select eight semi-monthly composites and derive indices

**Work to implement**

From Step 5's 24 annual composites, select days 1–15 and 16–month-end for each of the accepted four calendar months. Use start-inclusive/end-exclusive interval metadata. Do not recomposite the medians. Calculate NDVI, NDRE and NDMI from the five composite study bands, handling invalid denominators and -9999 nodata. Preserve clear_count and exclusion flags. B2 is for inspection, not an extra study feature. Median indices are a different calculation from indices of median bands; this workflow uses the latter.

**Expected output**

Eight intervals × eight study variables = 64 features per pixel, plus supporting quality layers.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


### Planned outputs

Expected outputs are the eight precomputed half-month composites corresponding to each accepted four-month window, three indices derived from median bands, and retained clear-observation-count layers. The primary stack contains 64 features per pixel; the five-band comparison contains 40. No second temporal median is applied. Implementation is not included yet.


## 12. Apply the missing-data policy

**Work to implement**

Start with a complete-data baseline requiring all 64 features. Count usable pixels per bin, not just acquisitions over the patch. Report losses by crop, field size and region. If necessary, evaluate a predefined short-gap policy separately; do not invent observed transitions across long gaps. Fit learned imputation on training data only.

**Expected output**

Accepted feature stacks and explicit missingness/exclusion records.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


### Planned outputs

Expected outputs are feature stacks meeting the chosen completeness policy, usable-pixel counts by bin and exclusions summarized by crop, field and region. Optional gap treatment must preserve reconstruction flags and be evaluated separately. These outputs define the usable sample pool for training tables. Implementation is not included yet.


## 13. Prepare tables and field-based partitions

**Work to implement**

Keep field_id, season_id, crop, source, region, window dates and quality as metadata. Use satellite features as model inputs and crop as target. Deduplicate fields before splitting; keep fragments, duplicate locations, pixels, windows and seasons of the same field together. Sample pixels across independent fields. Compare balanced sampling/class weights without treating correlated pixels as independent evidence.

**Expected output**

Training/validation tables and independent Benguet test fields.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:

**Pilot sampling:** Read `balanced_pilot/selected_pixels.csv` and use only locations marked 1 in the sampling masks. Keep every pixel and fragment in the same `partition_group` together; also verify geometry-based field deduplication before final splitting. Do not treat nearby pixels as independent fields. Report losses after quality and temporal screening before any rebalancing.


### Planned outputs

Expected outputs are pixel feature tables, crop targets, provenance and quality metadata, stable independent-field identifiers and train/validation/test assignments. All fragments, pixels, seasons and windows of the same field stay together. The final Benguet test fields remain independent of development choices. Implementation is not included yet.


## 14. Train and evaluate the thesis comparison

**Work to implement**

Compare external-only, limited Benguet-only, and external plus the same Benguet training fields. Train new Random Forests. Evaluate all on the same independent Benguet test fields using per-class precision/recall/F1, macro-F1, balanced accuracy and confusion matrices. Record growth-detector failures and eligible coverage. Do not tune using final test results.

**Expected output**

Reproducible model comparison, field-level sample counts and limitations.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


### Planned outputs

Expected outputs are the external-only, local-only and combined Random Forest models, their settings, per-crop metrics, confusion matrices and mapping/detector coverage reports. All comparisons use the same independent Benguet test fields. These results test whether external data improve performance; improvement is not assumed. Implementation is not included yet.


## 15. Document and package outputs

**Work to implement**

Record dataset versions, licenses, preprocessing parameters, detector settings, random seeds, split definitions and export task completion. Keep raw data separate from derived outputs. Document how field-season predictions relate to the intended Benguet map period.

**Expected output**

Auditable methodology and final artifact inventory.

**Your decisions and findings**

- Method and parameters:
- Checks performed:
- Results:
- Unresolved issues / next action:


### Planned outputs

Expected outputs are a methodology record, dataset revisions and licenses, preprocessing/detector parameters, seeds, split definitions, confirmed export records and an artifact inventory. The package documents how field-season predictions relate to the map reference period and identifies remaining limitations. Implementation is not included yet.


## Optional experiment: Field-level cloud coverage checks

**Work to implement**

Record clear-pixel fractions and supporting counts for each field/date. Validate the acceptance threshold on development fields; inspect residual clouds and reject unreliable observations. The current 80% threshold is exploratory, not a validated universal setting. Report exclusions and coverage by crop and region.

**Expected output**

Documented comparison, quality records and limitations.

**Your decisions and findings**

- Settings:
- Checks:
- Results:
- Limitations:


## Optional experiment: Feature comparison: 40 versus 64 inputs

**Work to implement**

Compare five bands × eight intervals (40 features) with five bands plus NDVI/NDRE/NDMI × eight intervals (64 features). Keep field partitions, windows, sample budgets and model selection procedures fixed. Select features only on development data; interpret correlated feature importance carefully.

**Expected output**

Documented comparison, quality records and limitations.

**Your decisions and findings**

- Settings:
- Checks:
- Results:
- Limitations:


## Optional experiment: Window sensitivity analysis

**Work to implement**

Keep the green-up-aligned four-calendar-month rule as the primary design. Test a predefined shift one semi-monthly bin earlier/later using eight consecutive calendar half-month bins. Label these as sensitivity variants, not the primary four-whole-month input. Report coverage changes and do not tune window shifts on the final Benguet test set.

**Expected output**

Documented comparison, quality records and limitations.

**Your decisions and findings**

- Settings:
- Checks:
- Results:
- Limitations:


## Optional experiment: Training-source comparison and per-crop reporting

**Work to implement**

Train external-only, limited Benguet-only, and external plus the same Benguet training fields. Evaluate on the same independent Benguet test fields. Report per-crop precision/recall/F1, macro-F1, balanced accuracy, confusion matrix, pixel/field counts, growth-detector failures and coverage. Keep every field together across partitions.

**Expected output**

Documented comparison, quality records and limitations.

**Your decisions and findings**

- Settings:
- Checks:
- Results:
- Limitations:


## Research and documentation

- [S4A downloads](https://huggingface.co/datasets/orion-ai-lab/S4A)
- [S4A repository and taxonomy](https://github.com/Orion-AI-Lab/S4A)
- [S4A paper](https://arxiv.org/abs/2204.00951)
- [L2A collection](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED)
- [Copernicus start of season: PPI amplitude method](https://www.copernicus.eu/en/access-data/copernicus-services-catalogue/start-season-date-2017-present-raster-10-m-europe-yearly)
- [Carrot Sentinel-2 development study](https://link.springer.com/article/10.1007/s11119-023-10083-z)

Literature supports methods, not field-specific planting/harvest dates. Proposed detector parameters require validation.

## Added research references

- [Crop-mapping review](https://doi.org/10.1080/10106049.2026.2690363)
- [Feature optimization](https://www.sciencedirect.com/science/article/pii/S1569843224005284)
- [Cloud quality and cross-year yield evaluation](https://ira.agroscope.ch/de-CH/Page/Einzelpublikation/Download?einzelpublikationId=56073)

These support experimental ideas; they do not validate the exact four-month design or crop-specific onset thresholds.


## Summary and limitations

Steps 1–8 prepare the balanced pilot and produce quality reports, timelines and candidate onset intervals. Steps 9–15 remain planned work.

The saved run checked on October 14, 2026 contains **960 composite records**: **874 passed** the initial coverage rule and **86 had low coverage. It contains **3,946 actual-date summary records** across **40 regions**. The primary detector reported candidates in **22 regions** and multi-cycle ambiguity in **10 regions**. These counts describe the saved run, not automatic updates after future reruns.

Cloud screening, coverage thresholds and detector parameters still need validation. Pixel counts are not independent-field counts. Green-up is not planting, and annual crop declarations do not confirm each selected cycle. No candidate is automatically accepted as a training season. Use the error reports, review records and Step 9 before proceeding.
